In [1]:
import os
import time
from collections import defaultdict

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from gymnasium import spaces
from IPython.display import display
from joblib import Parallel, delayed

from race_env import RaceEnv, GateSequenceMonitor, segment_intersection, point_segment_distance

In [2]:
COURSE_PATH = "../../ros2_ws/src/pylon_2026_world/purt_course.yaml"
CELL = 1.0

THRESHOLDS = {
    "(-12, -4)": [-12, -4],
    "(-10, -1)": [-10, -1],
    "(-8, -2)": [-8, -2],
    "(-6, -2)": [-6, -2],
    "(-4, -1)": [-4, -1],
}

STL_ORDER_WEIGHTS = list(range(0, 101, 10)) # Tune 10 different weights from 0 to 100; 0 is baseline
SEEDS = [123, 42, 999] # Retrain through 3 different seeds for each threshold

# Combination lengths: 5 x 11 x 3 = 165 combinations
# Settings were taken from gaussian-gates.ipynb
SETTINGS = {"MAX_STEPS": 400, "NOISE": 0.25, "DISCOUNT_FACTOR": 0.99, "ALPHA": 0.1, "EPSILON": 0.3,
            "test_rollouts": 200, "episodes": 3000}

LEARNED_LAP_RATE = 0.95 # for Pareto curve
N_JOBS = max(1, (os.cpu_count() or 2) - 2)

# STL gate-order monitor

In [3]:
def gate_robustness(env, a, b, gate_index):
    gate = env.GATES[gate_index]
    crossing = segment_intersection(a, b, gate["p1"], gate["p2"])
    if crossing is not None:
        _, u = crossing
        return min(u, 1.0 - u) * gate["length"]
    return -point_segment_distance(b, gate["p1"], gate["p2"])

STL_REWARDS = {"neutral": -1, "midpoint": 1000, "endpoint": 1000, "pylon_field": -300, "stl_order": 50}

In [4]:
class STLOrderMonitor(GateSequenceMonitor):
    def __init__(self, env, REWARDS=STL_REWARDS, ROUND_M=None, H_BINS=None):
        super().__init__(env, REWARDS)
        self.ROUND_M = ROUND_M
        self.H_BINS = H_BINS

        self.observation_space = spaces.Tuple((
            env.observation_space,
            spaces.Discrete(env.n_gates + 1),
            spaces.Box(-np.inf, np.inf, shape=(), dtype=np.float64)
        ))

        self.rho_avoidance = np.inf
        self.reset_order()
        self.observe()

    def targets(self):
        if self.q < self.env.n_gates:
            return self.q, (self.q + 1) % self.env.n_gates
        return 0, None

    def reset_stage(self):
        self.W = np.inf # find min
        self.H = -np.inf # find max

    def reset_order(self):
        self.reset_stage()
        self.rho_final = []
        self.order_history = []

    def order_robustness(self):
        values = self.rho_final + ([self.H] if np.isfinite(self.H) else [])
        return float(min(values)) if values else -np.inf

    def reset(self):
        self.reset_order()
        self.rho_avoidance = np.inf
        _, info = super().reset()
        return self.observe(), info

    def step(self, action, ignore_truncation=False):
        target, nxt = self.targets()
        q_before = self.q
        _, reward, done, truncated, info = super().step(action, ignore_truncation)
        a, b = info["prev_xy"], info["new_xy"]
        self.rho_avoidance = min(self.rho_avoidance, info["pylon_robustness"])

        rho_target = gate_robustness(self.env, a, b, target)
        rho_not_next = -gate_robustness(self.env, a, b, nxt) if nxt is not None else np.inf
        H_prev = self.H
        self.W = min(self.W, rho_not_next)
        self.H = max(self.H, min(rho_target, self.W))

        r_order = float(self.REWARDS["stl_order"]) * (self.H - H_prev) if np.isfinite(H_prev) else 0.0
        info["reward_parts"]["stl_order"] = r_order
        reward += r_order
        self.reward_history[-1] = reward
        W, H = self.W, self.H
        if self.q != q_before:
            self.rho_final.append(self.H)
            self.reset_stage()
        rho_order = self.order_robustness()

        self.order_history.append({"W": W, "H": H, "rho_order": rho_order})
        info.update({"W": W, "H": H, "rho_target": rho_target, "rho_not_next": rho_not_next,
                     "rho_order": rho_order, "rho_final": list(self.rho_final),
                     "order_history": self.order_history, "rho_avoidance": self.rho_avoidance})
        return self.observe(), reward, done, truncated, info

    def get_info(self):
        return {**super().get_info(), "rho_avoidance": self.rho_avoidance}

    def observe(self):
        H = self.H
        if self.ROUND_M is not None:
            H = float(np.round(H / self.ROUND_M) * self.ROUND_M)
        if self.H_BINS is not None:
            # Eq. 11: undefined and h <= θ1 share class 0; θ1 < h <= θ2 is 1; h > θ2 is 2
            H = 0 if not np.isfinite(H) else int(np.digitize(H, self.H_BINS, right=True))
        self.state = (int(self.env.state), int(self.q), H)
        self.state_history[-1] = self.state
        return self.state

# Training and evaluation

In [5]:
class VisitCounter(gym.Wrapper):
    def __init__(self, env):
        super().__init__(env)
        self.visited = set()

    def reset(self):
        obs, info = self.env.reset()
        self.visited.add(obs)
        return obs, info

    def step(self, action, ignore_truncation=False):
        out = self.env.step(action, ignore_truncation)
        self.visited.add(out[0])
        return out

def greedy_action(Q, state, n_actions):
    return int(np.argmax(Q[state])) if state in Q else np.random.randint(n_actions)
    
def evaluate_greedy(env, Q, n_rollouts):
    lap_steps, rho_order, rho_avoidance = [], [], []
    for _ in range(n_rollouts):
        state, _ = env.reset()
        while True:
            state, _, done, truncated, info = env.step(greedy_action(Q, state, env.action_space.n))
            if done or truncated:
                break
        if info["lap_finished"]:
            lap_steps.append(len(env.reward_history))
            rho_order.append(info["rho_order"])
            rho_avoidance.append(info["rho_avoidance"])
    finished = bool(lap_steps)
    return {"lap_rate": len(lap_steps) / n_rollouts,
            "lap_steps": float(np.mean(lap_steps)) if finished else np.nan,
            "rho_order": float(np.mean(rho_order)) if finished else np.nan,
            "rho_order_min": float(np.min(rho_order)) if finished else np.nan,
            "rho_avoidance": float(np.mean(rho_avoidance)) if finished else np.nan,
            "rho_avoidance_min": float(np.min(rho_avoidance)) if finished else np.nan}

def q_learning(env, discount_factor, alpha, epsilon, n_episodes):
    n_actions = env.action_space.n
    Q = defaultdict(lambda: np.random.random(n_actions))
    for _ in range(n_episodes):
        state, _ = env.reset()
        while True:
            action = np.random.randint(n_actions) if np.random.random() < epsilon else int(np.argmax(Q[state]))
            next_state, reward, done, truncated, _ = env.step(action)
            Q[state][action] += alpha * (reward + discount_factor * np.max(Q[next_state]) * (1 - done) - Q[state][action])
            state = next_state
            if done or truncated:
                break
    return Q

def train_one(label, h_bins, stl_order, seed):
    np.random.seed(seed)
    env = RaceEnv(COURSE_PATH, CELL_SIZE_M=CELL, MAX_STEPS=SETTINGS["MAX_STEPS"], NOISE=SETTINGS["NOISE"])
    monitor = STLOrderMonitor(env, REWARDS={**STL_REWARDS, "stl_order": stl_order}, H_BINS=h_bins)
    visits = VisitCounter(monitor)
    start = time.perf_counter()
    Q = q_learning(visits, SETTINGS["DISCOUNT_FACTOR"], SETTINGS["ALPHA"], SETTINGS["EPSILON"], SETTINGS["episodes"])
    train_s = time.perf_counter() - start
    return {"thresholds": label, "stl_order": stl_order, "seed": seed, "train_s": train_s,
            "states": len(visits.visited), **evaluate_greedy(monitor, Q, SETTINGS["test_rollouts"])}

# Run combination

In [6]:
jobs = [(label, bins, w, seed) for label, bins in THRESHOLDS.items() for w in STL_ORDER_WEIGHTS for seed in SEEDS]
print(f"Training {len(jobs)} runs: {len(THRESHOLDS)} threshold pairs x {len(STL_ORDER_WEIGHTS)} weights x "
      f"{len(SEEDS)} seeds on {N_JOBS} workers")

runs = []
start = time.perf_counter()
for r in Parallel(n_jobs=N_JOBS, return_as="generator_unordered")(delayed(train_one)(*job) for job in jobs):
    runs.append(r)
    print(f"[{len(runs):>3}/{len(jobs)}] {time.perf_counter() - start:>6.0f}s  θ {r['thresholds']:<10} "
          f"stl_order {r['stl_order']:<3} seed {r['seed']:<4}: lap {r['lap_rate']:.1%}, {r['lap_steps']:.1f} steps, "
          f"ρ_order {r['rho_order']:.2f} m, ρ_avoid {r['rho_avoidance']:.2f} m", flush=True)
print(f"Training took {time.perf_counter() - start:.0f}s wall ({sum(r['train_s'] for r in runs):.0f}s of training)")

Training 165 runs: 5 threshold pairs x 11 weights x 3 seeds on 20 workers
[  1/165]    278s  θ (-12, -4)  stl_order 20  seed 42  : lap 100.0%, 77.9 steps, ρ_order 4.83 m, ρ_avoid 0.22 m
[  2/165]    289s  θ (-12, -4)  stl_order 50  seed 999 : lap 100.0%, 74.7 steps, ρ_order 5.00 m, ρ_avoid 0.17 m
[  3/165]    307s  θ (-12, -4)  stl_order 30  seed 42  : lap 100.0%, 81.7 steps, ρ_order 3.42 m, ρ_avoid 0.28 m
[  4/165]    312s  θ (-12, -4)  stl_order 40  seed 42  : lap 100.0%, 78.3 steps, ρ_order 5.28 m, ρ_avoid 0.10 m
[  5/165]    317s  θ (-12, -4)  stl_order 30  seed 999 : lap 100.0%, 88.9 steps, ρ_order 4.18 m, ρ_avoid 0.21 m
[  6/165]    333s  θ (-12, -4)  stl_order 10  seed 999 : lap 100.0%, 77.9 steps, ρ_order 4.51 m, ρ_avoid 0.25 m
[  7/165]    343s  θ (-12, -4)  stl_order 10  seed 42  : lap 100.0%, 77.8 steps, ρ_order 4.21 m, ρ_avoid 0.47 m
[  8/165]    345s  θ (-12, -4)  stl_order 40  seed 123 : lap 100.0%, 87.8 steps, ρ_order 4.34 m, ρ_avoid 0.19 m
[  9/165]    361s  θ (-12, -4)

# Result table

In [7]:
lap_col = f"Lap @{SETTINGS['episodes']}"
theta_order = list(THRESHOLDS)

all_runs = (pd.DataFrame(runs)
            .rename(columns={"thresholds": "θ (m)", "seed": "Seed", "lap_rate": lap_col, "lap_steps": "Steps/lap",
                             "rho_order": "ρ_order (m)", "rho_order_min": "Worst ρ_order (m)",
                             "rho_avoidance": "ρ_avoid (m)", "rho_avoidance_min": "Worst ρ_avoid (m)",
                             "states": "States", "train_s": "Train s"})
            [["θ (m)", "stl_order", "Seed", lap_col, "Steps/lap", "ρ_order (m)", "Worst ρ_order (m)",
              "ρ_avoid (m)", "Worst ρ_avoid (m)", "States", "Train s"]])
all_runs = (all_runs.assign(_t=all_runs["θ (m)"].map(theta_order.index))
            .sort_values(["_t", "stl_order", "Seed"]).drop(columns="_t").reset_index(drop=True))

display(all_runs.style
        .format({lap_col: "{:.1%}", "Steps/lap": "{:.1f}", "ρ_order (m)": "{:.2f}", "Worst ρ_order (m)": "{:.2f}",
                 "ρ_avoid (m)": "{:.2f}", "Worst ρ_avoid (m)": "{:.2f}", "States": "{:.0f}", "Train s": "{:.0f}"},
                na_rep="-")
        .background_gradient(subset=[lap_col], cmap="Greens", vmin=0, vmax=1)
        .background_gradient(subset=["Steps/lap"], cmap="Blues_r")
        .background_gradient(subset=["ρ_order (m)"], cmap="Purples")
        .hide(axis="index"))

θ (m),stl_order,Seed,Lap @3000,Steps/lap,ρ_order (m),Worst ρ_order (m),ρ_avoid (m),Worst ρ_avoid (m),States,Train s
"(-12, -4)",0,42,100.0%,74.8,3.63,2.23,0.47,-0.97,6960,428
"(-12, -4)",0,123,100.0%,72.6,3.45,1.22,0.11,-1.35,6983,369
"(-12, -4)",0,999,100.0%,73.5,3.62,2.70,0.48,-0.13,7149,358
"(-12, -4)",10,42,100.0%,77.8,4.21,2.70,0.47,0.04,4989,326
"(-12, -4)",10,123,100.0%,79.3,4.69,2.55,0.15,-0.80,4594,363
"(-12, -4)",10,999,100.0%,77.9,4.51,2.55,0.25,-1.23,4133,316
"(-12, -4)",20,42,100.0%,77.9,4.83,3.46,0.22,-0.50,4570,257
"(-12, -4)",20,123,100.0%,88.1,4.31,2.23,0.41,-0.55,4334,422
"(-12, -4)",20,999,100.0%,85.6,3.45,2.23,0.19,-0.50,4216,386
"(-12, -4)",30,42,100.0%,81.7,3.42,1.88,0.28,-0.97,4112,290


In [8]:
summary = (all_runs.groupby(["θ (m)", "stl_order"], sort=False)
           .agg(**{lap_col: (lap_col, "mean"),
                   "Steps/lap": ("Steps/lap", "mean"), "Steps/lap sd": ("Steps/lap", "std"),
                   "ρ_order (m)": ("ρ_order (m)", "mean"), "ρ_order sd": ("ρ_order (m)", "std"),
                   "ρ_avoid (m)": ("ρ_avoid (m)", "mean"), "Worst ρ_avoid (m)": ("Worst ρ_avoid (m)", "min")})
           .reset_index())
summary["Reliable"] = ((summary[lap_col] >= LEARNED_LAP_RATE)
                       & summary["Steps/lap"].notna() & summary["ρ_order (m)"].notna())

def pareto(df):
    ok = df["Reliable"]
    front = pd.Series(False, index=df.index)
    for i in df.index[ok]:
        s, rho = df.at[i, "Steps/lap"], df.at[i, "ρ_order (m)"]
        o = df[ok & (df.index != i)]
        front[i] = not ((o["Steps/lap"] <= s) & (o["ρ_order (m)"] >= rho)
                        & ((o["Steps/lap"] < s) | (o["ρ_order (m)"] > rho))).any()
    return front

summary["Front (θ)"] = False
for label, g in summary.groupby("θ (m)", sort=False):
    summary.loc[g.index, "Front (θ)"] = pareto(g)
summary["Front (all)"] = pareto(summary)

display(summary.drop(columns=["Reliable"]).style
        .format({lap_col: "{:.1%}", "Steps/lap": "{:.1f}", "Steps/lap sd": "{:.1f}", "ρ_order (m)": "{:.2f}",
                 "ρ_order sd": "{:.2f}", "ρ_avoid (m)": "{:.2f}", "Worst ρ_avoid (m)": "{:.2f}"}, na_rep="-")
        .apply(lambda row: ["font-weight: bold" if row["Front (all)"] else "" for _ in row], axis=1)
        .hide(axis="index"))


def print_front(name, df):
    df = df.sort_values("Steps/lap")
    points = "  ->  ".join(f"w={r['stl_order']:.0f} ({r['Steps/lap']:.1f} steps, ρ_order {r['ρ_order (m)']:.2f} m)"
                           for _, r in df.iterrows())
    print(f"{name}: {points if points else 'no reliable configuration'}")

print("Pareto curves, fastest to most robust:")
for label, g in summary.groupby("θ (m)", sort=False):
    print_front(f"  θ {label}", g[g["Front (θ)"]])

θ (m),stl_order,Lap @3000,Steps/lap,Steps/lap sd,ρ_order (m),ρ_order sd,ρ_avoid (m),Worst ρ_avoid (m),Front (θ),Front (all)
"(-12, -4)",0,100.0%,73.6,1.1,3.56,0.10,0.35,-1.35,True,True
"(-12, -4)",10,100.0%,78.3,0.9,4.47,0.24,0.29,-1.23,True,False
"(-12, -4)",20,100.0%,83.9,5.3,4.20,0.70,0.28,-0.55,False,False
"(-12, -4)",30,100.0%,85.0,3.6,3.76,0.39,0.21,-1.32,False,False
"(-12, -4)",40,100.0%,89.3,11.8,4.91,0.50,0.15,-1.32,False,False
"(-12, -4)",50,100.0%,85.4,9.7,4.63,0.38,0.18,-0.97,False,False
"(-12, -4)",60,100.0%,85.4,3.6,5.13,0.20,0.17,-0.80,True,True
"(-12, -4)",70,100.0%,91.9,9.7,3.77,0.52,0.09,-0.97,False,False
"(-12, -4)",80,100.0%,107.3,26.4,5.15,0.09,0.25,-0.55,True,True
"(-12, -4)",90,85.7%,140.6,77.2,4.40,0.87,0.14,-1.33,False,False


Pareto curves, fastest to most robust:
  θ (-12, -4): w=0 (73.6 steps, ρ_order 3.56 m)  ->  w=10 (78.3 steps, ρ_order 4.47 m)  ->  w=60 (85.4 steps, ρ_order 5.13 m)  ->  w=80 (107.3 steps, ρ_order 5.15 m)
  θ (-10, -1): w=0 (73.5 steps, ρ_order 3.53 m)  ->  w=20 (76.7 steps, ρ_order 4.70 m)  ->  w=50 (84.6 steps, ρ_order 4.85 m)
  θ (-8, -2): w=10 (74.8 steps, ρ_order 4.46 m)  ->  w=40 (77.7 steps, ρ_order 4.68 m)  ->  w=100 (93.7 steps, ρ_order 4.72 m)
  θ (-6, -2): w=0 (73.9 steps, ρ_order 3.80 m)  ->  w=20 (74.7 steps, ρ_order 4.03 m)  ->  w=10 (76.0 steps, ρ_order 4.15 m)  ->  w=30 (77.2 steps, ρ_order 4.64 m)
  θ (-4, -1): w=20 (75.8 steps, ρ_order 4.44 m)  ->  w=90 (86.1 steps, ρ_order 4.55 m)
